In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("storageName", "adlsproybuckd01")

In [0]:
storageName = dbutils.widgets.get("storageName")

In [0]:
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-meta`
URL 'abfss://metastore@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas prueba del Data Lake';          
          """)


In [0]:
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-raw`
URL 'abfss://raw@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas prueba del Data Lake';
""")

In [0]:
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-bronze`
URL 'abfss://bronze@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas prueba del Data Lake';
""")

In [0]:
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-silver`
URL 'abfss://silver@${storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas prueba del Data Lake';
""")

In [0]:
spark.sql("""
CREATE EXTERNAL LOCATION IF NOT EXISTS `extl-golden`
URL 'abfss://golden@{storageName}.dfs.core.windows.net/'
WITH (STORAGE CREDENTIAL `credential`)
COMMENT 'Ubicación externa para las tablas prueba del Data Lake';
""")

In [0]:
spark.sql("""
DROP CATALOG IF EXISTS catalog_pry_final CASCADE;
""")

In [0]:
spark.sql(f"""
CREATE CATALOG IF NOT EXISTS catalog_pry_final
MANAGED LOCATION 'abfss://metastore@{storageName}.dfs.core.windows.net/'
COMMENT 'Catalogo para la arquitectura medallion del ambiente de dev';
""")

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS catalog_pry_final.bronze;")
spark.sql("CREATE SCHEMA IF NOT EXISTS catalog_pry_final.silver;")
spark.sql("CREATE SCHEMA IF NOT EXISTS catalog_pry_final.golden;")

###Tablas bronce

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.bronze.pasillo(
pasillo_id integer,
pasillo string,
ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/tablas/pasillo'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.bronze.departamentos(
departamento_id integer,
departamento string,
ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/tablas/departamento'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.bronze.productos(
producto_id integer,
producto_nombre string,
pasillo_id integer,
departamento_id integer,
ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/tablas/productos'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.bronze.ordenes(
orden_id integer,
user_id string,
eval_set string,
nro_orden integer,
orden_dwo integer,
hora_del_dia integer,
dias_desde_orden integer,
ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/tablas/ordenes'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.bronze.ordenes_prior(
orden_id integer,
producto_id string,
orden_agregado integer,
reordenado integer,
ingestion_date timestamp
)
USING DELTA
LOCATION 'abfss://bronze@{storageName}.dfs.core.windows.net/tablas/ordenes_prior'
""")

###Tablas Silver

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.silver.dim_productos(
producto_id integer,
producto_nombre string,
pasillo_id integer,
pasillo_nombre string,
departamento_id integer,
departamento_nombre string
)
USING DELTA
LOCATION 'abfss://silver@{storageName}.dfs.core.windows.net/tablas/dim_productos'
""")

###Tablas Golden

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.golden.dm_department_performance(
  departamento_nombre string,
  total_productos_vendidos long
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/tablas/dm_department_performance';
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.golden.dm_top_products(
  producto_nombre string,
  departamento_nombre string,
  total_vendido long
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/tablas/dm_top_products'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS catalog_pry_final.golden.user_profiling(
  user_id string,
  total_ordenes integer,
  prom_dias_entre_ordenes double
)
USING DELTA
LOCATION 'abfss://golden@{storageName}.dfs.core.windows.net/tablas/user_profiling'
""")